In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

In [2]:
print(f'Last run date: {dt.datetime.today()}')

Last run date: 2024-03-07 14:25:33.282012


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: 08_retro_scoring
Subtask: 05_get_tsp_from_db


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('with tblMax as\n'
 '(\n'
 'select\n'
 '\tbigAccountId,\n'
 '\tmax(concat(MonthOnBooks, bigAccountid, bigRunDateKeyId)) as MaxUnique\n'
 'from riskdb.dbo.tblReportCOStaticPools_StaticPool\n'
 'where MonthOnBooks<=72\n'
 'group by bigAccountId\n'
 '),\n'
 ' \n'
 '\n'
 '\n'
 'tblReportCOStaticPools_StaticPoolNew as\n'
 '(\n'
 'select\n'
 '\tconcat(MonthOnBooks, bigAccountid, bigRunDateKeyId) as uniqueC,\n'
 '\t*\n'
 'from riskdb.dbo.tblReportCOStaticPools_StaticPool\n'
 'where MonthOnBooks<=72\n'
 ')\n'
 ' \n'
 '\n'
 'select \n'
 '\tCONCAT(tblAccount.bigAccountId, tblAccount.bigDebtorId, 1) as UniqueID,\n'
 '\ttbltempstaticpool.bigAccountId,\n'
 '\ttblAccount.bigDebtorId,\n'
 '\t1 as bitDebtor,\n'
 '\ttblAccount.dtmStampCreation,\n'
 '\ttbltempstaticpool.dtmFunded,\n'
 "\tcase when tbltempstaticpool.LoanStatusCNCombined='Default' then 1 else 0 "
 'end as bitDefault,\n'
 '\ttblReportCOStaticPools_StaticPoolNew.MonthOnBooks,\n'
 '\ttblReportCOStaticPools_StaticPoolNew.RunningNetLoss,\n'
 

### Write into df

In [7]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# show
df

Wall time: 34.1 s


,UniqueID,bigAccountId,bigDebtorId,bitDebtor,dtmStampCreation,dtmFunded,bitDefault,MonthOnBooks,RunningNetLoss,AmtFinanced,...,fltDownCash,fltApprovedDownTotal,Payment,DTI,PTI,bitServiceContract,fltAdvance,strVehicleType,bitGap,DealerStampCreation
0,540765268178011,5407652,6817801,1,2020-11-19 13:54:29.723,2021-03-10,0,36,0.00,19347.14,...,1000.0,1000.0,469.58,0.346908,0.139358,1,1.092150,suv,0,2015-10-29 14:48:28.540
1,540888668193211,5408886,6819321,1,2020-11-20 14:05:40.133,2021-01-14,1,38,20567.25,22112.07,...,0.0,0.0,488.79,0.324816,0.048698,0,1.220344,suv,1,2014-06-25 16:00:43.403
2,541573568277731,5415735,6827773,1,2020-11-28 08:04:49.730,2021-01-21,0,38,0.00,23444.24,...,0.0,0.0,466.00,0.271003,0.093755,1,1.270458,auto,1,2012-11-21 13:38:42.487
3,541921668320781,5419216,6832078,1,2020-12-01 16:57:30.277,2021-01-14,0,38,0.00,18562.90,...,5000.0,2500.0,400.00,0.427233,0.100993,0,0.832685,suv,1,2016-10-05 14:04:47.870
4,542088568341181,5420885,6834118,1,2020-12-03 14:26:37.373,2021-02-12,0,37,0.00,13835.00,...,0.0,0.0,314.73,0.361740,0.050378,0,1.175182,auto,1,2017-02-06 15:26:30.390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84602,761809694585220,7618096,9458523,0,2024-02-28 17:07:29.973,2024-03-01,0,None,NaN,22344.09,...,2300.0,1000.0,672.85,0.386543,0.115552,0,1.152868,auto,1,2014-04-15 16:42:22.560
84603,761904294596610,7619042,9459662,0,2024-02-29 09:47:09.700,2024-03-04,0,None,NaN,25401.86,...,1000.0,500.0,718.98,0.461679,0.095553,0,1.198987,suv,1,2022-01-19 08:53:21.713
84604,762022294610900,7620222,9461091,0,2024-02-29 13:04:32.200,2024-03-06,0,None,NaN,14631.30,...,1000.0,1000.0,449.20,NaN,0.073121,0,1.213274,auto,0,2016-08-31 12:54:21.290
84605,762768094702900,7627680,9470291,0,2024-03-02 13:22:54.237,2024-03-04,0,None,NaN,26428.58,...,2000.0,1000.0,801.91,0.354109,0.107965,0,0.964333,auto,1,2008-03-07 08:50:17.213


### Lower column names and add suffix

In [8]:
%%time

# list_cols = [
# ]

# # subset
# df = df[list_cols]

# lower columns
df.columns = [f'{col.lower()}__app' for col in df.columns]

# show
df

Wall time: 2 ms


,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,dtmstampcreation__app,dtmfunded__app,bitdefault__app,monthonbooks__app,runningnetloss__app,amtfinanced__app,...,fltdowncash__app,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app
0,540765268178011,5407652,6817801,1,2020-11-19 13:54:29.723,2021-03-10,0,36,0.00,19347.14,...,1000.0,1000.0,469.58,0.346908,0.139358,1,1.092150,suv,0,2015-10-29 14:48:28.540
1,540888668193211,5408886,6819321,1,2020-11-20 14:05:40.133,2021-01-14,1,38,20567.25,22112.07,...,0.0,0.0,488.79,0.324816,0.048698,0,1.220344,suv,1,2014-06-25 16:00:43.403
2,541573568277731,5415735,6827773,1,2020-11-28 08:04:49.730,2021-01-21,0,38,0.00,23444.24,...,0.0,0.0,466.00,0.271003,0.093755,1,1.270458,auto,1,2012-11-21 13:38:42.487
3,541921668320781,5419216,6832078,1,2020-12-01 16:57:30.277,2021-01-14,0,38,0.00,18562.90,...,5000.0,2500.0,400.00,0.427233,0.100993,0,0.832685,suv,1,2016-10-05 14:04:47.870
4,542088568341181,5420885,6834118,1,2020-12-03 14:26:37.373,2021-02-12,0,37,0.00,13835.00,...,0.0,0.0,314.73,0.361740,0.050378,0,1.175182,auto,1,2017-02-06 15:26:30.390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84602,761809694585220,7618096,9458523,0,2024-02-28 17:07:29.973,2024-03-01,0,None,NaN,22344.09,...,2300.0,1000.0,672.85,0.386543,0.115552,0,1.152868,auto,1,2014-04-15 16:42:22.560
84603,761904294596610,7619042,9459662,0,2024-02-29 09:47:09.700,2024-03-04,0,None,NaN,25401.86,...,1000.0,500.0,718.98,0.461679,0.095553,0,1.198987,suv,1,2022-01-19 08:53:21.713
84604,762022294610900,7620222,9461091,0,2024-02-29 13:04:32.200,2024-03-06,0,None,NaN,14631.30,...,1000.0,1000.0,449.20,NaN,0.073121,0,1.213274,auto,0,2016-08-31 12:54:21.290
84605,762768094702900,7627680,9470291,0,2024-03-02 13:22:54.237,2024-03-04,0,None,NaN,26428.58,...,2000.0,1000.0,801.91,0.354109,0.107965,0,0.964333,auto,1,2008-03-07 08:50:17.213


### Save

In [9]:
%%time

# save
str_filename = 'df_scored_accounts.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 6.94 s


### Upload to s3

In [10]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 1.82 s


### Clean-up

In [11]:
os.remove(str_local_path)